In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score

from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import AgglomerativeClustering

In [13]:
from google.colab import files
import pandas as pd

print("Please upload your CSV file.")
uploaded = files.upload()

# Assuming you uploaded a single CSV file, get its name
file_name = None
for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))
  file_name = fn
  break # Assuming only one file will be uploaded

if file_name:
    try:
        # Read the uploaded CSV file into a pandas DataFrame
        df = pd.read_csv(file_name)
        print("File loaded successfully as a CSV file.")
        # Display the first 5 rows of the DataFrame and its information
        display(df.head())
        display(df.info())
    except Exception as e:
        print(f"Failed to load file as CSV. Error: {e}")
        df = None
else:
    print("No file was uploaded.")
    df = None

Please upload your CSV file.


Saving online_retail_combined.csv to online_retail_combined.csv
User uploaded file "online_retail_combined.csv" with length 94248800 bytes
File loaded successfully as a CSV file.


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  object 
 1   StockCode    1067371 non-null  object 
 2   Description  1062989 non-null  object 
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  object 
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 65.1+ MB


None

### Data Cleaning and Preprocessing

In [15]:
if df is not None:
    # Drop rows with any missing values
    df.dropna(inplace=True)

    # Convert 'InvoiceDate' to datetime objects
    df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

    # Remove canceled orders (Invoice usually starts with 'C' for cancellations)
    df = df[~df['Invoice'].astype(str).str.contains('C', na=False)]

    # Remove rows with negative Quantity or UnitPrice
    df = df[df['Quantity'] > 0]
    df = df[df['Price'] > 0]

    # Calculate TotalPrice for each item
    df['TotalPrice'] = df['Quantity'] * df['Price']

    # Display cleaned data info
    print("\n--- Cleaned DataFrame Info ---")
    display(df.info())
    display(df.describe())
else:
    print("Cannot perform data cleaning: DataFrame 'df' is None. Please check the file loading step.")


--- Cleaned DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
Index: 805549 entries, 0 to 1067370
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      805549 non-null  object        
 1   StockCode    805549 non-null  object        
 2   Description  805549 non-null  object        
 3   Quantity     805549 non-null  int64         
 4   InvoiceDate  805549 non-null  datetime64[ns]
 5   Price        805549 non-null  float64       
 6   Customer ID  805549 non-null  float64       
 7   Country      805549 non-null  object        
 8   TotalPrice   805549 non-null  float64       
dtypes: datetime64[ns](1), float64(3), int64(1), object(4)
memory usage: 61.5+ MB


None

,Quantity,InvoiceDate,Price,Customer ID,TotalPrice
count,805549.000000,805549,805549.000000,805549.000000,805549.000000
mean,13.290522,2011-01-02 10:24:44.106814464,3.206561,15331.954970,22.026505
min,1.000000,2009-12-01 07:45:00,0.001000,12346.000000,0.001000
25%,2.000000,2010-07-07 12:08:00,1.250000,13982.000000,4.950000
50%,5.000000,2010-12-03 15:10:00,1.950000,15271.000000,11.850000
75%,12.000000,2011-07-28 13:05:00,3.750000,16805.000000,19.500000
max,80995.000000,2011-12-09 12:50:00,10953.500000,18287.000000,168469.600000
std,143.634088,NaN,29.199173,1696.737039,224.041928
